# Code mẫu — Regularization

Regularization thay đổi bài toán tối ưu bằng cách phạt những trọng số lớn. Ba ví dụ dưới đây tách riêng từng ý: phạt làm hệ số co lại hay bị bỏ hẳn, đơn vị đo ảnh hưởng tới mức phạt ra sao, và chọn mức phạt thế nào mà không học lén từ validation.

## Ví dụ 1: Hệ số nhỏ bị co lại, hay bị bỏ hẳn?

Xét ba feature trực chuẩn, tức các cột có độ dài 1 và vuông góc với nhau, với hệ số least squares là `[3, 0.5, −1.2]`. Trong ví dụ này, Ridge dùng `0.5 × lỗi bình phương + 0.5 × λ × ||w||²`, còn Lasso dùng `0.5 × lỗi bình phương + λ × ||w||₁`. Cần ghi rõ quy ước như vậy, vì chỉ lệch một hệ số 2 là các ngưỡng tính tay sẽ sai.

In [ ]:
import numpy as np

z = np.array([3., 0.5, -1.2])  # Bằng X.T @ y, khi X.T @ X là ma trận đơn vị.
for strength in [0.5, 1., 2.]:
    ridge_w = z / (1 + strength)
    lasso_w = np.sign(z) * np.maximum(np.abs(z) - strength, 0)
    print("lambda =", strength,
          "Ridge:", ridge_w.round(3).tolist(),
          "Lasso:", lasso_w.round(3).tolist())
assert np.allclose(np.sign(z) * np.maximum(np.abs(z) - 1, 0), [2., 0., -0.2])

lambda = 0.5 Ridge: [2.0, 0.333, -0.8] Lasso: [2.5, 0.0, -0.7]
lambda = 1.0 Ridge: [1.5, 0.25, -0.6] Lasso: [2.0, 0.0, -0.2]
lambda = 2.0 Ridge: [1.0, 0.167, -0.4] Lasso: [1.0, 0.0, -0.0]


Ở λ = 1, Ridge chia cả ba hệ số cho 2, nên hệ số nào cũng nhỏ đi nhưng không cái nào về 0. Lasso thì trừ đi đúng 1 ở mỗi hệ số: hệ số 0.5 bị kéo hẳn về 0, còn hai hệ số kia chỉ nhỏ đi. Đây là lý do Lasso hay được dùng để chọn feature. Hai công thức gọn như vậy chỉ đúng khi các cột trực chuẩn. Khi các cột tương quan với nhau, Ridge không còn co mọi hệ số theo cùng một tỉ lệ, và Lasso phải giải bằng một thuật toán tối ưu.

## Ví dụ 2: Chỉ đổi km thành mét cũng làm đổi mức phạt

Bốn mẫu có khoảng cách `[0, 1, 2, 3]` km, và chi phí `[0, 2, 4, 6]` đơn vị. Ridge với `alpha=5` phạt hệ số của khoảng cách. Ta so ba cách: dùng km để nguyên, dùng mét để nguyên, và chuẩn hóa bên trong Pipeline.

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X_km = np.array([[0.], [1.], [2.], [3.]])
y = np.array([0., 2., 4., 6.])
query_km = np.array([[1.5], [4.]])
scaled_predictions = []
for unit, factor in [("km", 1.), ("m", 1000.)]:
    X = X_km * factor
    raw = Ridge(alpha=5.).fit(X, y)
    pipeline = make_pipeline(StandardScaler(), Ridge(alpha=5.)).fit(X, y)
    prediction = pipeline.predict(query_km * factor)
    scaled_predictions.append(prediction)
    print(unit, "hệ số thô quy về mỗi km:", round(float(raw.coef_[0] * factor), 4))
    print(unit, "dự đoán có scaler:", prediction.round(4).tolist())
print("hai pipeline khớp:", np.allclose(*scaled_predictions))
assert np.allclose(*scaled_predictions)

km hệ số thô quy về mỗi km: 1.0
km dự đoán có scaler: [3.0, 5.2222]
m hệ số thô quy về mỗi km: 2.0
m dự đoán có scaler: [3.0, 5.2222]
hai pipeline khớp: True


Với km, nghiệm có hệ số `10 / (5 + 5) = 1`, tức bị phạt kéo xuống một nửa so với giá trị đúng là 2. Với mét, hệ số quy ra mỗi km lại gần 2, vì con số hệ số theo mét rất nhỏ, nên gần như không bị phạt. Cùng một dữ liệu, chỉ đổi đơn vị mà model học ra hai kết quả khác nhau. Hai Pipeline có scaler thì cho cùng dự đoán, vì scaler học mean và std từ train, nên việc đổi đơn vị bị triệt tiêu ngay ở bước chuẩn hóa. Đó là lý do phải chuẩn hóa feature trước khi dùng regularization. Điểm 4 km nằm ngoài dữ liệu, nên ở đây nó chỉ dùng để kiểm tính bất biến theo đơn vị, chứ không đo chất lượng dự đoán.

## Ví dụ 3: Chọn Ridge hay Lasso khi bảng có nhiều cột nhiễu

Hai feature đầu tạo ra target, còn tám feature sau là nhiễu độc lập. Ta giữ riêng 30 mẫu làm test ngay từ đầu, và chọn alpha trên 90 mẫu còn lại, với cùng ba fold cho cả hai model. Thứ tự hệ số trong output chính là thứ tự của mười cột đầu vào.

In [ ]:
import numpy as np
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(22)
X = rng.normal(size=(120, 10))
y = 2 * X[:, 0] - X[:, 1] + rng.normal(0, 0.6, len(X))
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.25, random_state=3
)
folds = list(KFold(3, shuffle=True, random_state=5).split(X_dev))
searches = {}
for name, estimator, alphas in [
    ("Ridge", Ridge(), [0.1, 1., 10., 100.]),
    ("Lasso", Lasso(max_iter=10000), [0.01, 0.03, 0.1, 0.3]),
]:
    pipe = Pipeline([("scale", StandardScaler()), ("model", estimator)])
    # Scaler được fit lại trong từng fold. Không chuẩn hóa X_dev trước khi đưa vào GridSearchCV.
    search = GridSearchCV(pipe, {"model__alpha": alphas}, cv=folds,
                          scoring="neg_mean_squared_error", n_jobs=1)
    search.fit(X_dev, y_dev)
    searches[name] = search
    weights = search.best_estimator_["model"].coef_
    print(name, "alpha:", search.best_params_["model__alpha"],
          "CV MSE:", round(float(-search.best_score_), 4),
          "hệ số khác 0:", int(np.count_nonzero(weights)))
    print("hệ số:", weights.round(3).tolist())

Ridge alpha: 1.0 CV MSE: 0.4412 hệ số khác 0: 10
hệ số: [1.891, -0.908, -0.155, -0.029, 0.036, -0.109, 0.06, 0.04, -0.061, 0.012]
Lasso alpha: 0.1 CV MSE: 0.4033 hệ số khác 0: 4
hệ số: [1.773, -0.844, -0.041, -0.0, 0.0, -0.013, 0.0, 0.0, -0.0, 0.0]


Hai lưới alpha khác nhau là có chủ đích. Ridge trong scikit-learn phạt trên tổng lỗi bình phương, còn Lasso phạt trên lỗi chia cho `2n`, nên cùng một con số alpha không mang cùng ý nghĩa. Các hệ số in ra là theo cột đã chuẩn hóa, tức một đơn vị là một độ lệch chuẩn trong train, chứ không phải đơn vị gốc. Ridge giữ cả mười hệ số khác 0, dù tám cột sau chỉ là nhiễu. Lasso được chọn alpha = 0.1 và chỉ còn bốn hệ số khác 0, gồm hai cột thật cùng hai cột nhiễu. Cross-validation chọn theo chất lượng dự đoán, nên không có gì bảo đảm Lasso giữ đúng hai cột tạo ra target.

Chọn phương án theo điểm CV trước, rồi mới chấm test, và chỉ chấm một lần. Kiểm thêm một baseline đoán bằng trung bình của tập dev, để biết MSE của model có thật sự đáng kể hay không.

In [ ]:
chosen_name = max(searches, key=lambda name: searches[name].best_score_)
chosen = searches[chosen_name].best_estimator_
pred = chosen.predict(X_test)
baseline = np.full(len(y_test), y_dev.mean())
print("chọn bằng CV:", chosen_name)
print("test MSE baseline / mô hình:",
      round(mean_squared_error(y_test, baseline), 4),
      round(mean_squared_error(y_test, pred), 4))
print("train MSE:", round(mean_squared_error(y_dev, chosen.predict(X_dev)), 4))
print("4 cặp thật / dự đoán:", np.column_stack([y_test, pred])[:4].round(3).tolist())
assert chosen["scale"].n_samples_seen_ == len(X_dev)
assert pred.shape == y_test.shape and np.isfinite(pred).all()

chọn bằng CV: Lasso
test MSE baseline / mô hình: 3.6342 0.3202
train MSE: 0.3545
4 cặp thật / dự đoán: [[-0.266, 0.409], [-3.743, -3.919], [0.47, 1.154], [-2.692, -3.018]]


Lasso thắng theo CV, và trên test nó đạt MSE 0.3202, thấp hơn rất nhiều so với 3.6342 của baseline. MSE train là 0.3545, lần này lại hơi cao hơn test. Hai tập có mẫu và nhiễu khác nhau, nên không có quy tắc nào buộc test luôn tệ hơn train. Sau khi CV đã chọn cấu hình, scaler được fit lại trên cả 90 mẫu dev, và dòng assert kiểm đúng điều đó. Nếu xem kết quả test rồi quay lại đổi alpha, thì test đã tham gia vào việc chọn, và con số 0.3202 không còn là đánh giá độc lập nữa.